In [26]:
import torch
import pandas as pd
import numpy as np
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import os
import time
import matplotlib.pyplot as plt
from sklearn import metrics
from torch.optim.lr_scheduler import LambdaLR
from tqdm import tqdm
from efficient_kan import KAN
import collections.abc as c
import os

In [9]:
def createSmarts():
    smarts = {'alkane':'[CX4;H0,H1,H2,H4]',
                   'methyl':'[CH3]',
                   'alkene':'[CX3]=[CX3]',
                   'alkyne':'[CX2]#C',
                   'alcohols':'[#6][OX2H]',
                   'amines':'[NX3;H2,H1;!$(NC=O)]', 
                   'nitriles':'[NX1]#[CX2]', 
                   'aromatics':'[$([cX3](:*):*),$([cX2+](:*):*)]',
                   'alkyl halides':'[#6][F,Cl,Br,I]', 
                   'esters':'[#6][CX3](=O)[OX2H0][#6]', 
                   'ketones':'[#6][CX3](=O)[#6]',
                   'aldehydes':'[CX3H1](=O)[#6]', 
                   'carboxylic acids':'[CX3](=O)[OX2H1]', 
                   'ether': '[OD2]([#6])[#6]',
                   'acyl halides':'[CX3](=[OX1])[F,Cl,Br,I]',
                   'amides':'[NX3][CX3](=[OX1])[#6]',
                   'nitro':'[$([NX3](=O)=O),$([NX3+](=O)[O-])][!#8]',
                   'heterocyclic': '[!#6;!R0]',
                   'aryl chlorides': '[Cl][c]',
                   'carboxylic esters': '[CX3;$([R0][#6]),$([H1R0])](=[OX1])[OX2][#6;!$(C=[O,N,S])]',
                   'alkyl aryl ethers': '[OX2](c)[CX4;!$(C([OX2])[O,S,#7,#15,F,Cl,Br,I])]',
                   'phenols': '[OX2H][c]',
                   'trifluoromethyl':'[#6](F)(F)F',
                   'isocyanate':'[*][N]=C=O',
                   'phosphate':'OP(=O)(O)O' ,
                   'fluorinated':'[CX4,cX3][F]'}
    return smarts;

#Creates hashmaps where data is further retrieved by functional group name and its whether its training or test set



In [24]:
def createHashmaps(test_ratio: float = 0.3,
                    batch_size: int = 128):
    
    '''
    Creates hashmaps containing FTIR data.

    Parameters
    ----------

    test_ratio : float, default 0.3
        Ratio of test dataset.
    
    batch_size : int, default 128
        Size of the batch.
    '''

    func_groups_data, func_groups_datasets, func_groups_dataloaders = {}, {}, {}
    for data_directory in os.listdir(func_groups_path):
        print(data_directory)
        #if data_directory=="carboxylic acids" or data_directory=="carboxylic esters":
        #    return func_groups_data, func_groups_datasets, func_groups_dataloaders
        dataset = FunctionalGroupsDataset(data_directory, convert_to='absorbance')

        training_dataset, test_dataset = random_split(dataset, [1 - test_ratio, test_ratio], torch.Generator())

        func_groups_dataloaders[data_directory] = {'training': DataLoader(training_dataset, batch_size = batch_size, shuffle = True), 'test' : DataLoader(test_dataset, batch_size = batch_size, shuffle = False)}

    return func_groups_data, func_groups_datasets, func_groups_dataloaders


In [1]:
##tu wczytujemy nowy dataset

def createHashmaps1(test_ratio: float = 0.3,
                    batch_size: int = 128):
    
    '''
    Creates hashmaps containing FTIR data.

    Parameters
    ----------

    test_ratio : float, default 0.3
        Ratio of test dataset.
    
    batch_size : int, default 128
        Size of the batch.
    '''

    func_groups_data, func_groups_datasets, func_groups_dataloaders = {}, {}, {}
    for data_directory in os.listdir(func_groups_path):
        print(data_directory)
        if data_directory=='alcohols' or data_directory=='alkane'  or data_directory=='aryl chlorides' or data_directory=='carboxylic acids' or data_directory=='carboxylic esters' or data_directory=='alkyl aryl ethers'  or data_directory=='alkyl halides' or data_directory=='alkene' or data_directory=='amines' or data_directory=='nitriles' or data_directory=='esters' or data_directory=='ketones' or data_directory=='nitro'  or data_directory=='phenols' or data_directory=='methyl' or data_directory=='fluorinated' or data_directory=='amides'  or data_directory=='ether' or data_directory=='heterocyclic' or data_directory=='aromatics' :
            
       
            dataset = FG3Dataset(data_directory, convert_to='absorbance')

            training_dataset, test_dataset = random_split(dataset, [1 - test_ratio, test_ratio], torch.Generator())

            func_groups_dataloaders[data_directory] = {'training': DataLoader(training_dataset, batch_size = batch_size, shuffle = True), 'test' : DataLoader(test_dataset, batch_size = batch_size, shuffle = False)}

    return func_groups_data, func_groups_datasets, func_groups_dataloaders

In [1]:
def createHashmapsCombined(test_ratio: float = 0.3,
                    batch_size: int = 128):
    
    '''
    Creates hashmaps containing FTIR data.

    Parameters
    ----------

    test_ratio : float, default 0.3
        Ratio of test dataset.
    
    batch_size : int, default 128
        Size of the batch.
    '''

    func_groups_data, func_groups_datasets, func_groups_dataloaders = {}, {}, {}
    for data_directory in os.listdir(func_groups_path):
        print(data_directory)
        if data_directory=='alcohols' or data_directory=='alkene' or data_directory=='amines' or data_directory=='nitriles' or data_directory=='esters' or data_directory=='ketones' or data_directory=='nitro'  or data_directory=='phenols' or data_directory=='methyl' or data_directory=='fluorinated' or data_directory=='amides' or data_directory=='aromatics' :
            
            datasetNIST = FunctionalGroupsDataset(data_directory, convert_to='absorbance')
            datasetNEW = FG3Dataset(data_directory, convert_to='absorbance')
 
            training_datasetNEW, test_dataset = random_split(datasetNEW, [1 - test_ratio, test_ratio], torch.Generator())
            training_datasetCombined = torch.utils.data.ConcatDataset([training_datasetNEW, datasetNIST])

            
            func_groups_dataloaders[data_directory] = {'training': DataLoader(training_datasetCombined, batch_size = batch_size, shuffle = True), 'test' : DataLoader(test_dataset, batch_size = batch_size, shuffle = False)}

    return func_groups_data, func_groups_datasets, func_groups_dataloaders

In [25]:
class FunctionalGroupsDataset(Dataset):
    '''
    PyTorch compatible dataset of functional groups files

    Parameters
    ----------

    func_group : str
        Name of the functional group to retrieve data from.
    
    convert_to : str, default None
        Converts all data to specified data type.

    '''
    def __init__(self, func_group: str, convert_to: str = None) -> None:
        self.convert_to = convert_to
        self.func_group_number = np.where(func_group_names.values == func_group)[0][0]
        self.main_dir =  os.path.join('D:\\SHAPE\\1VII\\functional-cnn-main\\ALL')
        self.func_group = func_group

        #NIST IDs that passed preprocessing
        preprocessed_data_dir = pd.DataFrame(os.listdir(os.path.join(self.main_dir, 'preprocessed_data')))


        if func_group not in os.listdir(os.path.join(self.main_dir, 'functional_groups')):
            raise ValueError(f'{func_group} is not present in our database.')
        else:
            #All NIST IDs of specific functional group
            func_group_data_dir = pd.DataFrame(os.listdir(os.path.join(self.main_dir, 'functional_groups', func_group)))

        #NIST IDs of specific functional group that passed preprocessing
        to_sample = pd.merge(preprocessed_data_dir, func_group_data_dir, on = [0, 0], how = 'outer', indicator = True).query('_merge=="left_only"')[0]

        #Equinumerous dataset of preprocessed functional group NIST IDs and shuffled from every other functional groups
        if len(to_sample) < len(func_group_data_dir):
            func_group_data_dir = func_group_data_dir.sample(len(to_sample))
            self.data = pd.concat([to_sample.sample(len(to_sample)), func_group_data_dir], axis = 0)
        else:
            self.data = pd.concat([to_sample.sample(len(func_group_data_dir)), func_group_data_dir], axis = 0)

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        file_path = os.path.join(self.main_dir, 'preprocessed_data', self.data.iloc[index][0])
        file = pd.read_csv(file_path)
        spectra_type = file['spectraType'][0]



        #Converts data
        # if not self.convert_to:
        #     spectra = torch.nan_to_num(torch.tensor(file['y'].values, requires_grad=True)).to(torch.float)

        # elif self.convert_to.lower() not in ('absorbance', 'absorbancja', 'transmittance', 'transmitancja'):
        #     raise ValueError(f'Cant convert to {self.convert_to}.')

        # elif self.convert_to.lower() != str(spectra_type).lower():
        #     spectra = torch.nan_to_num(torch.tensor(np.abs(1 - file['y'].values), requires_grad=True)).to(torch.float)
        
        # else:
        #     spectra = torch.nan_to_num(torch.tensor(file['y'].values, requires_grad=True)).to(torch.float)

        if not self.convert_to:
            spectra = torch.nan_to_num(torch.tensor(file['y'].values, requires_grad=True)).to(torch.float)

        elif self.convert_to.lower() not in ('absorbance', 'absorbancja', 'transmittance', 'transmitancja'):
            raise ValueError(f'Cant convert to {self.convert_to}.')
        
        elif str(spectra_type).lower() not in ('converted_to_absorbance', "absorbance"):
            spectra = torch.nan_to_num(torch.tensor(np.abs(1 - file['y'].values), requires_grad=True)).to(torch.float)
        
        else:
            spectra = torch.nan_to_num(torch.tensor(file['y'].values, requires_grad=True)).to(torch.float)

        #Reshapes it as required
        spectra = spectra.reshape(1, 1, 3106)
        
        #Prevents unknown problem with NaN from before
        func_group = torch.nan_to_num(torch.tensor(file['funcGroups'].values[self.func_group_number], requires_grad=True)).to(torch.float)

        return spectra, func_group
    
    def get_nist_id(self, index):
        return self.data.iloc[index][0]

In [13]:
# Evaluation of prediction quality on the TRAINING set. The function is adapted to the
# func_groups_dataloaders dataloader for the NIST dataset.
def training_acc(model, group,iter=0,dirpath="C:\\Users\\Tomek_\\Desktop\\cc\\res"):

    model.eval()
    actual = []
    pred = []
    with torch.no_grad():
        lista_grup = np.zeros(1)
        list_func = np.zeros(1)

        for y_batch, funcGroup in func_groups_dataloaders[group]['training']:
            y_batch, funcGroup = y_batch.to(device), funcGroup.to(device)

            outputs = model(y_batch)
            outputs = torch.Tensor.cpu(outputs).detach().numpy()
            funcGroup = torch.Tensor.cpu(funcGroup).detach().numpy()

            pred.extend(outputs.reshape(-1).round())
            actual.extend(funcGroup.reshape(-1))

            for i in range(outputs.shape[0]):
                predicted_well = (np.round(outputs[i]) == funcGroup[i])*1 * funcGroup[i]
                    
                lista_grup+=predicted_well
                list_func+=funcGroup[i]

        df = pd.concat([pd.DataFrame([group]), pd.DataFrame(lista_grup/list_func), pd.DataFrame(lista_grup), pd.DataFrame(list_func)], axis=1, ignore_index=True)
        df.columns = ['funcGroup', 'prediction acc', 'predicted well', 'number of groups']

        confusion_matrix = metrics.confusion_matrix(actual, pred)
        trueNegative=confusion_matrix[0][0]
        truePositive=confusion_matrix[1][1]
        falsePositive=confusion_matrix[0][1]
        falseNegative=confusion_matrix[1][0]

        cfMatrixString="EM: "+str(truePositive)+" "+str(trueNegative)+" "+str(falsePositive)+" "+str(falseNegative)+" ";
        
        sensitivity=truePositive/(truePositive+falseNegative);
        specificity=trueNegative/(trueNegative+falsePositive)
        print("The sensitivity is: ",  str(round(sensitivity*100, 2))  ," % whereas the specificity is ", str(round(specificity*100, 2))," %")
        cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix = confusion_matrix, display_labels = [0, 1])
        cm_display.plot()
        accu=(truePositive+trueNegative)/(truePositive+trueNegative+falsePositive+falseNegative)
        opis="Iteration "+str(iter)+" training. Group "+group+"."+cfMatrixString+" The sensitivity is: "+  str(round(sensitivity*100, 2))  +" % whereas the specificity is: "+ str(round(specificity*100, 2))+" %  accuracy wynosi: "+str(round(accu*100, 2))+" %\r\n"
       
        global summary_description
        summary_description+=opis;
        
        plt.savefig(dirpath+"\\training_"+str(iter)+".png")
        plt.show()
        return df
        
#training_acc(model, group)

In [14]:
# Evaluation of prediction quality on the TEST set. The function is adapted to the
# func_groups_dataloaders dataloader for the NIST dataset.
def test_acc(model, group,iter=0,dirpath="C:\\Users\\Tomek_\\Desktop\\cc\\res"):
    model.eval()
    actual = []
    pred = []
    with torch.no_grad():
        lista_grup = np.zeros(1)
        list_func = np.zeros(1)

        for y_batch, funcGroup in func_groups_dataloaders[group]['test']:
            y_batch, funcGroup = y_batch.to(device), funcGroup.to(device)

            outputs = model(y_batch)
            outputs = torch.Tensor.cpu(outputs).detach().numpy()
            funcGroup = torch.Tensor.cpu(funcGroup).detach().numpy()

            pred.extend(outputs.reshape(-1).round())
            actual.extend(funcGroup.reshape(-1))

            for i in range(outputs.shape[0]):
                predicted_well = (np.round(outputs[i]) == funcGroup[i])*1# * funcGroup[i]
                
                lista_grup+=predicted_well
                list_func+=1#funcGroup[i]

        df = pd.concat([pd.DataFrame([group]), pd.DataFrame(lista_grup/list_func), pd.DataFrame(lista_grup), pd.DataFrame(list_func)], axis=1, ignore_index=True)
        df.columns = ['funcGroups', 'prediction acc', 'predicted well', 'number of samples']

        confusion_matrix = metrics.confusion_matrix(actual, pred)
        trueNegative=confusion_matrix[0][0]
        truePositive=confusion_matrix[1][1]
        falsePositive=confusion_matrix[0][1]
        falseNegative=confusion_matrix[1][0]
        cfMatrixString="EM: "+str(truePositive)+" "+str(trueNegative)+" "+str(falsePositive)+" "+str(falseNegative)+" ";
     
        sensitivity=truePositive/(truePositive+falseNegative);
        specificity=trueNegative/(trueNegative+falsePositive)
        print("The sensitivity is: ",  str(round(sensitivity*100, 2))  ," % whereas the specificity is: ", str(round(specificity*100, 2))," %")
        cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix = confusion_matrix, display_labels = [0, 1])
        cm_display.plot()
        accu=(truePositive+trueNegative)/(truePositive+trueNegative+falsePositive+falseNegative)
        opis="Iteration "+str(iter)+" test. Group "+group+". "+cfMatrixString+" The sensitivity is: "+  str(round(sensitivity*100, 2))  +" % whereas the specificity is: "+ str(round(specificity*100, 2))+" %  accuracy wynosi: "+str(round(accu*100, 2))+" %\r\n"
             
        global summary_description
        summary_description+=opis;
        
        plt.savefig(dirpath+"\\test_"+str(iter)+".png")
        plt.show()
        return df

summary_description="";
#test_acc(model, group)

In [16]:
## Testing of predictions on the test set of the new dataset (ENFSI)

def test_acc1(model, group,iter=0,dirpath="C:\\Users\\Tomek_\\Desktop\\cc\\res"):

    model.eval()
    actual = []
    pred = []
    with torch.no_grad():
        lista_grup = np.zeros(1)
        list_func = np.zeros(1)

        for y_batch, funcGroup in func_groups_dataloadersN[group]['test']:
            y_batch, funcGroup = y_batch.to(device), funcGroup.to(device)

            outputs = model(y_batch)
            outputs = torch.Tensor.cpu(outputs).detach().numpy()
            funcGroup = torch.Tensor.cpu(funcGroup).detach().numpy()

            pred.extend(outputs.reshape(-1).round())
            actual.extend(funcGroup.reshape(-1))


            for i in range(outputs.shape[0]):
                predicted_well = (np.round(outputs[i]) == funcGroup[i])*1# * funcGroup[i]
                
                lista_grup+=predicted_well
                list_func+=1#funcGroup[i]


        df = pd.concat([pd.DataFrame([group]), pd.DataFrame(lista_grup/list_func), pd.DataFrame(lista_grup), pd.DataFrame(list_func)], axis=1, ignore_index=True)
        df.columns = ['funcGroups', 'prediction acc', 'predicted well', 'number of groups']

        confusion_matrix = metrics.confusion_matrix(actual, pred)
        trueNegative=confusion_matrix[0][0]
        truePositive=confusion_matrix[1][1]
        falsePositive=confusion_matrix[0][1]
        falseNegative=confusion_matrix[1][0]
        cfMatrixString="EM: "+str(truePositive)+" "+str(trueNegative)+" "+str(falsePositive)+" "+str(falseNegative)+" ";
        print(cfMatrixString)
        sensitivity=truePositive/(truePositive+falseNegative);
        specificity=trueNegative/(trueNegative+falsePositive)
        print("The sensitivity is: ",  str(round(sensitivity*100, 2))  ," % whereas the specificity is: ", str(round(specificity*100, 2))," %")
        cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix = confusion_matrix, display_labels = [0, 1])
        cm_display.plot()
        accu=(truePositive+trueNegative)/(truePositive+trueNegative+falsePositive+falseNegative)
        opis="Iteration "+str(iter)+" test. Group "+group+". "+cfMatrixString+" The sensitivity is: "+  str(round(sensitivity*100, 2))  +" % whereas the specificity is: "+ str(round(specificity*100, 2))+" %  accuracy wynosi: "+str(round(accu*100, 2))+" %\r\n"
        print(opis)
        
        global summary_description
        summary_description+=opis;
        
        plt.savefig(dirpath+"\\test_"+str(iter)+".png")
        plt.show()
        return df


summary_description="";
#test_acc(model, group)

In [17]:
## Testing of predictions on the training set of the new dataset (ENFSI)

def train_acc1(model, group,iter=0,dirpath="C:\\Users\\Tomek_\\Desktop\\cc\\res"):

    model.eval()
    actual = []
    pred = []
    with torch.no_grad():
        lista_grup = np.zeros(1)
        list_func = np.zeros(1)

        for y_batch, funcGroup in func_groups_dataloadersN[group]['training']:
            y_batch, funcGroup = y_batch.to(device), funcGroup.to(device)

            outputs = model(y_batch)
            outputs = torch.Tensor.cpu(outputs).detach().numpy()
            funcGroup = torch.Tensor.cpu(funcGroup).detach().numpy()

            pred.extend(outputs.reshape(-1).round())
            actual.extend(funcGroup.reshape(-1))


            for i in range(outputs.shape[0]):
                predicted_well = (np.round(outputs[i]) == funcGroup[i])*1 * funcGroup[i]
                
                lista_grup+=predicted_well
                list_func+=funcGroup[i]


        df = pd.concat([pd.DataFrame([group]), pd.DataFrame(lista_grup/list_func), pd.DataFrame(lista_grup), pd.DataFrame(list_func)], axis=1, ignore_index=True)
        df.columns = ['funcGroups', 'prediction acc', 'predicted well', 'number of samples']

        confusion_matrix = metrics.confusion_matrix(actual, pred)
        trueNegative=confusion_matrix[0][0]
        truePositive=confusion_matrix[1][1]
        falsePositive=confusion_matrix[0][1]
        falseNegative=confusion_matrix[1][0]
        cfMatrixString="EM: "+str(truePositive)+" "+str(trueNegative)+" "+str(falsePositive)+" "+str(falseNegative)+" ";
     
        sensitivity=truePositive/(truePositive+falseNegative);
        specificity=trueNegative/(trueNegative+falsePositive)
        print("The sensitivity is: ",  str(round(sensitivity*100, 2))  ," % whereas the specificity is: ", str(round(specificity*100, 2))," %")
        cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix = confusion_matrix, display_labels = [0, 1])
        cm_display.plot()
        accu=(truePositive+trueNegative)/(truePositive+trueNegative+falsePositive+falseNegative)
        opis="Iteration "+str(iter)+" training. Group "+group+". "+cfMatrixString+" The sensitivity is: "+  str(round(sensitivity*100, 2))  +" % whereas the specificity is: "+ str(round(specificity*100, 2))+" %  accuracy wynosi: "+str(round(accu*100, 2))+" %\r\n"
     
        
        global summary_description
        summary_description+=opis;
        
        plt.savefig(dirpath+"\\test_"+str(iter)+".png")
        plt.show()
        return df


summary_description="";
#test_acc(model, group)

In [23]:
#for test/debug purpose only
def repeatabilityTest(method: str,
                      loss_func,
                      num_iters: int = 10,
                      num_epochs: int = 30):

    '''
    Test to check the repeatability of the model over all functional groups.

    Parameters
    ----------

    method : str
        Name to distinguish results from each other.
    
    loss_func : Any
        Loss function.
    
    num_iters : int, default 10
        Number of iterations for each functional group.

    num_epochs : int, default 30
        Number of epochs for each model training.
    '''

    for group in smarts.keys():
        
        group_WS = group.replace(' ', '')
        save_dirpath = os.path.join('.', 'results', method, group_WS)

    
        if not os.path.exists(save_dirpath):
            os.makedirs(save_dirpath)
            print("Directory created successfully!")
        else:
            print("Directory already exists!")


        for iteration in range(num_iters):

            start = time.time()

            _, _, func_groups_dataloaders = createHashmaps()

            
            model = train(num_epochs = num_epochs, 
                        loss_func = loss_func,
                        group = group,
                        data_loaders = func_groups_dataloaders,
                        weight_decay = 1e-2,
                        lambda_lr = 0.9,
                        learning_rate = 1e-2,
                        kan_layers = [1, 1, 1],
                        seed = 42,
                        plot = True,
                        save = False,
                        disable_verbose = False,
                        iter = 0,
                        save_dirpath = '.')
            
            print(trainingMetrics(model = model, 
                                   group = group,
                                   data_loaders = func_groups_dataloaders,
                                   iter = iteration,
                                   save_dirpath = save_dirpath))
            
            print(testMetrics(model = model, 
                                group = group,
                                data_loaders = func_groups_dataloaders,
                                iter = iteration,
                                save_dirpath = save_dirpath))
            
            global summary

            print(summary)
            print(f'Time per iteration: {time.time() - start}')

            with open(os.path.join(save_dirpath, 'stats.txt'), 'w') as f:
                f.write(summary)